# Minimal SPECTER Embeddings (full_text only)

- Nutzt nur `full_text`
- Leere `full_text`-Einträge werden übersprungen
- Nimmt maximal 100 Einträge
- Nutzt GPU falls verfügbar, sonst CPU

In [23]:
%pip install -q transformers torch python-dotenv tqdm

In [3]:
import json
import os
from pathlib import Path

import torch
from dotenv import load_dotenv
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
hf_token = "PASTE_YOUR_HF_TOKEN_HERE"
print("HF token loaded:", bool(hf_token and hf_token != "PASTE_YOUR_HF_TOKEN_HERE"))

data_path = Path("/content/drive/MyDrive/Data/ADS/queried_publications_lang_translated_openai_tokenized.json")
print(f"Using file: {data_path}")
if not data_path.exists():
    raise FileNotFoundError(f"File not found: {data_path}")

texts = []
rows = []

with data_path.open("r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        obj = json.loads(line)
        full_text = (obj.get("full_text") or "").strip()
        if not full_text:
            continue
        texts.append(full_text)
        rows.append(obj)
        if len(texts) >= 100:
            break

print(f"Loaded {len(texts)} non-empty full_text entries.")

HF token loaded: False
Using file: /content/drive/MyDrive/Data/ADS/queried_publications_lang_translated_openai_tokenized.json
Loaded 100 non-empty full_text entries.


In [27]:
model_name = "allenai/specter2_base"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print("CUDA available:", torch.cuda.get_device_name(0))
else:
    print("CUDA not available -> running on CPU")

hf_kwargs = {"token": hf_token} if hf_token else {}
tokenizer = AutoTokenizer.from_pretrained(model_name, **hf_kwargs)
model = AutoModel.from_pretrained(model_name, **hf_kwargs).to(device)
model.eval()

print(f"Device: {device}")

CUDA available: Tesla T4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: allenai/specter2_base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda


In [28]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

batch_size = 16
all_embeddings = []
batch_starts = range(0, len(texts), batch_size)

with torch.no_grad():
    for i in tqdm(batch_starts, desc="Embedding batches"):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        outputs = model(**encoded)
        emb = mean_pooling(outputs.last_hidden_state, encoded["attention_mask"])
        all_embeddings.append(emb.cpu())

embeddings = torch.cat(all_embeddings, dim=0) if all_embeddings else torch.empty((0, 0))
print("Embeddings shape:", tuple(embeddings.shape))

Embedding batches:   0%|          | 0/7 [00:00<?, ?it/s]

Embeddings shape: (100, 768)


In [32]:
import json
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/Data/ADS/embeddings")
output_dir.mkdir(parents=True, exist_ok=True)

embeddings_list = embeddings.tolist()
if len(rows) != len(embeddings_list):
    raise ValueError(f"Length mismatch: rows={len(rows)} embeddings={len(embeddings_list)}")

enriched_rows = []
for row, emb in zip(rows, embeddings_list):
    new_row = dict(row)
    new_row["embedding"] = emb
    enriched_rows.append(new_row)

drive_output = output_dir / "queried_publications_lang_translated_openai_tokenized_specter2_100.jsonl"
with drive_output.open("w", encoding="utf-8") as f:
    for record in enriched_rows:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

print(f"Saved to Drive: {drive_output}")
print(f"Rows with embeddings: {len(enriched_rows)}")

Saved to Drive: /content/drive/MyDrive/Data/ADS/embeddings/queried_publications_lang_translated_openai_tokenized_specter2_100.jsonl
Rows with embeddings: 100
Open/download the file from Google Drive (or use Google Drive Desktop sync locally).


In [33]:
with drive_output.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        record = json.loads(line)
        print(f"Row {i}:")
        print(f"  Bibcode: {record['Bibcode']}")
        print(f"  Title: {record['Title'][:80]}...")
        print(f"  Embedding shape: {len(record['embedding'])}")
        print()

Row 0:
  Bibcode: 1995PASP..107..803U
  Title: Unified Schemes for Radio-Loud Active Galactic Nuclei...
  Embedding shape: 768

Row 1:
  Bibcode: 1972gcpa.book.....W
  Title: Gravitation and Cosmology: Principles and Applications of the General Theory of ...
  Embedding shape: 768

Row 2:
  Bibcode: 1984ucp..book.....W
  Title: General Relativity...
  Embedding shape: 768



In [5]:
import pandas as pd
from pathlib import Path

# Pfade definieren
base_dir = Path(r"C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation")
h5_path = base_dir / "LSPO_v1.h5"
parquet_path = base_dir / "LSPO_v1.parquet"

try:
    print(f"Lade {h5_path}...")
    # DataFrame aus HDF5 laden (Key ist 'names')
    df = pd.read_hdf(h5_path, key='names')
    
    print("Erfolgreich geladen!")
    print(f"Shape: {df.shape}")
    print("Spalten:", df.columns.tolist())
    
    # Anzeigen
    if 'display' in globals():
        display(df.head())
    else:
        print(df.head())
    
    # Als Parquet speichern
    print(f"Speichere als {parquet_path}...")
    df.to_parquet(parquet_path)
    print("Erfolgreich gespeichert.")

except Exception as e:
    print(f"Fehler: {e}")


Lade C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation\LSPO_v1.h5...
Erfolgreich geladen!
Shape: (554962, 6)
Spalten: ['@path', 'title', 'abstract', 'author', 'aff', 'block']
                  @path                                              title  \
0  /0000-0003-1178-1001     Class of ghost-free non-Abelian gauge theories   
2  /0000-0001-5974-7043  Weak nonleptonic decays of charmed hadrons in ...   
3  /0000-0003-2257-3080  Target asymmetry in inclusive photoproduction ...   
4  /0000-0003-2257-3080     A space-time description of quarks and hadrons   
5  /0000-0001-9638-3082  Observation of spatial and temporal variations...   

                                            abstract          author  \
0  We discuss a class of non-Abelian gauge theori...  Frenkel, Josif   
2  We analyze the two-body weak nonleptonic decay...      Branco, G.   
3  We study the target asymmetry in inclusive pio...  Craigie, N. S.   
4  A more concrete for